In [1]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Load env variables and connect to database
load_dotenv("../.env")
DB_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DB_URL)

# Read raw data
df = pd.read_sql("SELECT * FROM ames_housing_project", engine)


def clean_data(df):
    df = df.copy()

    # Remove strange outliers:
    # very big houses but low price (does not make sense)
    if 'total_living_area' in df.columns and 'sale_price' in df.columns:
        mask = (df['total_living_area'] > 5000) & (df['sale_price'] < 400000)
        df = df.drop(df[mask].index, axis=0)

    # Remove IDs and unused column
    # they do not help the model
    df = df.drop(columns=['order_id', 'pid', 'garage_yr_blt'], errors='ignore')

    # Garage: missing means no garage
    # numeric → 0, categorical → 'None'
    g_num = [c for c in df.columns if 'garage' in c and df[c].dtype != 'object']
    g_cat = [c for c in df.columns if 'garage' in c and df[c].dtype == 'object']
    df[g_num] = df[g_num].fillna(0)
    df[g_cat] = df[g_cat].fillna('None')

    # Basement: same logic as garage
    b_num = [c for c in df.columns if 'bsmt' in c and df[c].dtype != 'object']
    b_cat = [c for c in df.columns if 'bsmt' in c and df[c].dtype == 'object']
    df[b_num] = df[b_num].fillna(0)
    df[b_cat] = df[b_cat].fillna('None')

    # Electrical has very few missing → fill with most common value
    if 'electrical' in df.columns:
        df['electrical'] = df['electrical'].fillna(df['electrical'].mode()[0])

    # Masonry: if missing, assume no material
    if 'mas_vnr_area' in df.columns:
        df['mas_vnr_area'] = df['mas_vnr_area'].fillna(0)
    if 'mas_vnr_type' in df.columns:
        df['mas_vnr_type'] = df['mas_vnr_type'].fillna('None')

    # Lot frontage depends on neighborhood
    # first fill using neighborhood mean, then global mean
    if 'neighborhood' in df.columns and 'lot_frontage' in df.columns:
        df['lot_frontage'] = df.groupby('neighborhood')['lot_frontage'].transform(
            lambda x: x.fillna(x.mean())
        )
        df['lot_frontage'] = df['lot_frontage'].fillna(df['lot_frontage'].mean())

    return df


# Run cleaning
df_cleaned = clean_data(df)

# Save cleaned data
df_cleaned.to_sql('cleaned_housing_data', engine, if_exists='replace', index=False)

print(f"Rows after cleaning: {len(df_cleaned)}")

Rows after cleaning: 2930
